# 독립 아이템 축 + N/V 블록 dropout M2 — Dunnhumby seed 42

직전 진단에서 ID+N과 ID+V는 각각 도움이 됐지만, 두 축을 항상 같이 학습하면 Top-10에서 간섭이 발생했습니다. 이 실행은 구조를 다음과 같이 제한합니다.

- 학습: Dunnhumby 1~683일
- 평가: 684~690일의 신규 상품
- 표현: ID 64차원 + 거래활동 4차원 + 거래당 가치 4차원
- 아이템 N/V 좌표는 ID 아이템 임베딩에서 투영하지 않고, 같은 BPR 손실에서 독립적으로 공동 학습
- 학습 중 ID는 항상 사용하고 N/V 축은 각각 50% 확률로 독립적으로 끄거나 켬
- 학습 상태: ID-only, ID+N, ID+V, ID+N+V가 각각 25%
- 평가 시에는 N/V 두 축을 모두 사용: `S_ID + 0.1S_N + 0.1S_V`
- 고정: binary graph, uniform negative sampling, 표본 가중 없는 plain BPR, 100 epoch
- 새로 학습: 이번 M2 하나만. M1@64 seed 42는 동일 분할·설정일 때만 기존 결과 재사용

이 실행은 역사적 개발구간의 seed 42 탐색이며 유의성을 주장하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = 'cdbbc64141120f572a10f4a978c3fa5929bd0dd1'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA
print('코드 고정 완료:', REVIEWED_SHA)

In [ ]:
import json
import torch
from lightgcn_clv_gatefree_lowdim_independent_dropout import (
    configure_independent_dropout_run,
    preflight_summary,
    run_independent_dropout_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_independent_dropout_run(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_gatefree_lowdim_independent_dropout_historical_screen_v1'
    ),
    baseline_result_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_repeatshare_historical_backtest_v1'
    ),
)
summary = preflight_summary(cfg)
assert summary['m2']['independent_item_axis_coordinates'] is True
assert summary['m2']['fixed_per_axis_budget'] == 0.10
assert summary['m2']['axis_keep_probability'] == 0.50
assert summary['m2']['training_state_probabilities'] == {
    'ID_only': 0.25,
    'ID_plus_activity': 0.25,
    'ID_plus_transaction_value': 0.25,
    'full': 0.25,
}
assert summary['m2']['evaluation_score_formula'] == 'S_ID + 0.1*S_N + 0.1*S_V'
assert summary['fixed']['graph'] == 'binary'
assert summary['fixed']['negative_sampling'] == 'uniform'
assert summary['fixed']['sample_weighting'] is False
assert summary['fixed']['validation_or_epoch_selection'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_independent_dropout_screen(cfg)

In [ ]:
from IPython.display import display

comparison = result_df.attrs['comparison'].copy()
reading = dict(result_df.attrs['screening_reading'])
paths = dict(result_df.attrs['result_paths'])
display_df = result_df.copy()
display_df.attrs = {}

print('절대지표:')
display(display_df.sort_values('model_id'))

core_metrics = [
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10',
    'coverage@10', 'n_distinct@10', 'top10_share@10',
]
print('M1@64 대비 핵심 변화:')
display(comparison[comparison['metric'].isin(core_metrics)].sort_values('metric'))
print('탐색 판독:', reading)
print('결과 파일:', paths)